## How `kubectl` actually works

When you typed `kubectl run hello --image=nginx:alpine`, you used the **`kubectl`** client — one binary on your laptop. But `kubectl` didn't run the pod. It built an object and sent it to the API server.

```
 kubectl  --HTTPS REST-->  kube-apiserver  -->  etcd
 (reads ~/.kube/config)    authn · authz         (cluster state)
                           validate · write
```

Lock in a few things:

- **`kubectl` is stateless.** It reads `~/.kube/config` for the API server URL and credentials, then makes HTTPS calls. Anyone with that file can drive the cluster.
- **Every component talks to the API server.** Scheduler, kubelet, controllers, `kubectl` — none touch `etcd` directly. The API server is the only writer.
- **Auth happens at the API server.** *Who you are* comes from your kubeconfig credential; *what you may do* is decided by **RBAC** (notebook 10).
- **The watch model.** The scheduler and kubelet don't *poll* — they open a streaming watch ("tell me when a pod changes") and react. That's how the cluster stays responsive.

### Anatomy of a `kubectl` command

Most commands read: `kubectl <verb> <resource> [name] [flags]`

- `kubectl get pods` — list all pods in the current namespace
- `kubectl describe pod hello` — verbose details + recent events
- `kubectl logs hello` — the container's stdout/stderr
- `kubectl exec -it hello -- sh` — a shell inside the pod
- `kubectl get all -n kube-system` — common kinds in another namespace

Conventions worth memorising now:

- **Names pluralise + alias.** `pod`, `pods`, `po` are the same. `kubectl api-resources` lists them.
- **Namespace flag.** Pods live in namespaces; default is `default`. `-n <ns>` targets one; `-A` means all.
- **Output format.** `-o wide`, `-o yaml`, `-o json`, `-o jsonpath=...` — critical for the exam.

Two views to bridge when troubleshooting: what the **api-server** thinks the state is, and what the **node** actually runs. On our map, that's the arrow from **Client** into the api-server — the single door everything passes through.